# TimeDataModel — Walkthrough

This notebook walks through the core features of `timedatamodel`: a domain-agnostic
time series data model with rich metadata, geolocation support, and seamless
conversion to NumPy, pandas, and polars.

## 1. Enums

`timedatamodel` provides three `StrEnum` types that serialize naturally to strings.

In [1]:
from timedatamodel import Frequency, DataType, StorageType

# Frequency uses ISO 8601 duration strings
print("Hourly:", Frequency.PT1H)
print("Daily: ", Frequency.P1D)
print("Monthly:", Frequency.P1M)

# StrEnum — direct string comparison works
assert Frequency.PT1H == "PT1H"

# DataType classifies the nature of the data
print("\nAll data types:", [dt.value for dt in DataType])

# StorageType describes how overlapping instances are stored
print("Storage types:", [st.value for st in StorageType])

Hourly: PT1H
Daily:  P1D
Monthly: P1M

All data types: ['MEASUREMENT', 'ESTIMATION', 'FORECAST', 'SCENARIO', 'SYNTHETIC', 'CLIMATE', 'ACTUAL']
Storage types: ['FLAT', 'OVERLAPPING']


## 2. Location

Two location types cover point and area use cases:
- **`GeoLocation`** — a single (latitude, longitude) point
- **`GeoArea`** — a Shapely polygon with optional name

In [2]:
from timedatamodel import GeoLocation, GeoArea

# A point location — Oslo, Norway
oslo = GeoLocation(latitude=59.91, longitude=10.75)
print("Oslo:", oslo)

Oslo: GeoLocation(latitude=59.91, longitude=10.75)


In [3]:
# Coordinates are validated on construction
try:
    GeoLocation(latitude=999, longitude=0)
except ValueError as e:
    print("Caught:", e)

Caught: latitude must be between -90 and 90, got 999


In [4]:
# A polygon area — simplified southern Norway region
# Coordinates are (latitude, longitude) pairs
no1 = GeoArea.from_coordinates(
    [(58.0, 6.0), (58.0, 12.0), (62.0, 12.0), (62.0, 6.0)],
    name="NO1",
)

print("Area name:  ", no1.name)
print("Bounding box:", no1.bounds)  # (min_lon, min_lat, max_lon, max_lat)
print("Centroid:   ", no1.centroid)

Area name:   NO1
Bounding box: (6.0, 58.0, 12.0, 62.0)
Centroid:    GeoLocation(latitude=60.0, longitude=9.0)


## 3. Resolution

`Resolution` pairs a `Frequency` with a timezone. Timezones are validated
against the standard library's `zoneinfo`.

In [5]:
from timedatamodel import Resolution

hourly_cet = Resolution(frequency=Frequency.PT1H, timezone="Europe/Oslo")
daily_utc = Resolution(frequency=Frequency.P1D)  # defaults to UTC
monthly = Resolution(frequency=Frequency.P1M)

print("Hourly CET:", hourly_cet)
print("ZoneInfo:  ", hourly_cet.tz)
print("Timedelta: ", hourly_cet.to_timedelta())
print()
print("Monthly is calendar-based:", monthly.is_calendar_based)
print("Monthly timedelta:        ", monthly.to_timedelta())  # None for calendar-based

Hourly CET: Resolution(frequency=<Frequency.PT1H: 'PT1H'>, timezone='Europe/Oslo')
ZoneInfo:   Europe/Oslo
Timedelta:  1:00:00

Monthly is calendar-based: True
Monthly timedelta:         None


## 4. Metadata

`Metadata` bundles descriptive information about a time series. Units are stored as
plain strings (easy to serialize) and resolved to `pint.Unit` on demand via the
`pint_unit` property.

In [6]:
from timedatamodel import Metadata

meta = Metadata(
    unit="MW",
    data_type=DataType.ACTUAL,
    location=oslo,
    name="power",
    description="Hourly power generation",
    attributes={"source": "example", "fuel": "wind"},
)

print("Unit string:", meta.unit)
print("Pint unit:  ", meta.pint_unit)
print("Data type:  ", meta.data_type)
print("Location:   ", meta.location)
print("Attributes: ", meta.attributes)

Unit string: MW
Pint unit:   megawatt
Data type:   ACTUAL
Location:    GeoLocation(latitude=59.91, longitude=10.75)
Attributes:  {'source': 'example', 'fuel': 'wind'}


## 5. Creating a TimeSeries

A `TimeSeries` combines a `Resolution`, `Metadata`, and the actual data.
It can be constructed from parallel lists or from a list of `DataPoint` tuples.

In [7]:
from datetime import datetime, timedelta, timezone
from timedatamodel import TimeSeries, DataPoint

# Construction from parallel lists
base = datetime(2024, 6, 1, tzinfo=timezone.utc)
timestamps = [base + timedelta(hours=i) for i in range(24)]
values = [120.0 + i * 3.5 for i in range(24)]
values[5] = None  # simulate a missing reading

ts = TimeSeries(
    hourly_cet,
    meta,
    timestamps=timestamps,
    values=values,
)

print(f"TimeSeries with {len(ts)} data points")
print(f"Resolution: {ts.resolution.frequency} / {ts.resolution.timezone}")

TimeSeries with 24 data points
Resolution: PT1H / Europe/Oslo


In [8]:
ts

Frequency,PT1H
Timezone,Europe/Oslo
Unit,MW
Data type,ACTUAL
Location,"59.91°N, 10.75°E"
Description,Hourly power generation
timestamp,power
2024-06-01 00:00:00+00:00,120.0
2024-06-01 01:00:00+00:00,123.5
2024-06-01 02:00:00+00:00,127.0
…,…


In [8]:
# Construction from DataPoint named tuples
data_points = [
    DataPoint(datetime(2024, 1, 1, tzinfo=timezone.utc), 10.0),
    DataPoint(datetime(2024, 1, 1, 1, tzinfo=timezone.utc), 12.5),
    DataPoint(datetime(2024, 1, 1, 2, tzinfo=timezone.utc), None),
]

ts2 = TimeSeries(hourly_cet, data=data_points)
print("From DataPoints:", list(ts2))

From DataPoints: [DataPoint(timestamp=datetime.datetime(2024, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), value=10.0), DataPoint(timestamp=datetime.datetime(2024, 1, 1, 1, 0, tzinfo=datetime.timezone.utc), value=12.5), DataPoint(timestamp=datetime.datetime(2024, 1, 1, 2, 0, tzinfo=datetime.timezone.utc), value=None)]


In [16]:
ts

TimeSeries(resolution=Resolution(frequency=<Frequency.PT1H: 'PT1H'>, timezone='Europe/Oslo'), metadata=Metadata(unit='MW', data_type=<DataType.ACTUAL: 'ACTUAL'>, location=GeoLocation(latitude=59.91, longitude=10.75), name='power', description='Hourly power generation', storage_type=<StorageType.FLAT: 'FLAT'>, attributes={'source': 'example', 'fuel': 'wind'}))

## 6. Sequence Protocol

`TimeSeries` supports `len()`, indexing, slicing, iteration, and truthiness.

In [17]:
# Indexing returns a DataPoint named tuple
first = ts[0]
print(f"First point: {first.timestamp} -> {first.value}")

# Slicing returns a list of DataPoints
print(f"\nFirst 3 points:")
for dp in ts[:3]:
    print(f"  {dp.timestamp}  {dp.value}")

# Iteration
print(f"\nLast 3 points:")
for dp in ts[-3:]:
    print(f"  {dp.timestamp}  {dp.value}")

# Truthiness
print(f"\nHas data: {bool(ts)}")
print(f"Empty:    {bool(TimeSeries(hourly_cet))}")

First point: 2024-06-01 00:00:00+00:00 -> 120.0

First 3 points:
  2024-06-01 00:00:00+00:00  120.0
  2024-06-01 01:00:00+00:00  123.5
  2024-06-01 02:00:00+00:00  127.0

Last 3 points:
  2024-06-01 21:00:00+00:00  193.5
  2024-06-01 22:00:00+00:00  197.0
  2024-06-01 23:00:00+00:00  200.5

Has data: True
Empty:    False


## 7. Export to NumPy

`to_numpy()` returns values as a `float64` array where `None` becomes `NaN`.

In [18]:
import numpy as np

arr = ts.to_numpy()
print(f"Shape: {arr.shape}, dtype: {arr.dtype}")
print(f"Values: {arr[:8]}")
print(f"Contains NaN at index 5: {np.isnan(arr[5])}")

Shape: (24,), dtype: float64
Values: [120.  123.5 127.  130.5 134.    nan 141.  144.5]
Contains NaN at index 5: True


## 8. Export to pandas

`to_pandas_dataframe()` returns a `DataFrame` with a `DatetimeIndex`. The column
name comes from `metadata.name` (or defaults to `"value"`).

In [19]:
df_pd = ts.to_pandas_dataframe()
print(f"Columns: {df_pd.columns.tolist()}")
print(f"Index type: {type(df_pd.index).__name__}")
print()
df_pd.head(8)

Columns: ['power']
Index type: DatetimeIndex



,power
timestamp,
2024-06-01 00:00:00+00:00,120.0
2024-06-01 01:00:00+00:00,123.5
2024-06-01 02:00:00+00:00,127.0
2024-06-01 03:00:00+00:00,130.5
2024-06-01 04:00:00+00:00,134.0
2024-06-01 05:00:00+00:00,NaN
2024-06-01 06:00:00+00:00,141.0
2024-06-01 07:00:00+00:00,144.5


## 9. Export to polars

`to_polars_dataframe()` returns a polars `DataFrame` with `"timestamp"` and value
columns. polars is lazy-imported, so it's only needed if you call this method.

In [20]:
df_pl = ts.to_polars_dataframe()
print(f"Columns: {df_pl.columns}")
print()
df_pl.head(8)

Columns: ['timestamp', 'power']



timestamp,power
"datetime[μs, UTC]",f64
2024-06-01 00:00:00 UTC,120.0
2024-06-01 01:00:00 UTC,123.5
2024-06-01 02:00:00 UTC,127.0
2024-06-01 03:00:00 UTC,130.5
2024-06-01 04:00:00 UTC,134.0
2024-06-01 05:00:00 UTC,NaN
2024-06-01 06:00:00 UTC,141.0
2024-06-01 07:00:00 UTC,144.5


## 10. Round-Trip Conversion

You can convert back from pandas or polars DataFrames to a `TimeSeries`.

In [21]:
# pandas round-trip
ts_from_pd = TimeSeries.from_pandas(df_pd, hourly_cet, meta)
print(f"From pandas: {len(ts_from_pd)} points")
print(f"  First: {ts_from_pd[0]}")
print(f"  Missing value preserved: {ts_from_pd[5].value is None}")

# polars round-trip
ts_from_pl = TimeSeries.from_polars(df_pl, hourly_cet, meta)
print(f"\nFrom polars: {len(ts_from_pl)} points")
print(f"  First: {ts_from_pl[0]}")
print(f"  Missing value preserved: {ts_from_pl[5].value is None}")

From pandas: 24 points
  First: DataPoint(timestamp=datetime.datetime(2024, 6, 1, 0, 0, tzinfo=datetime.timezone.utc), value=120.0)
  Missing value preserved: True

From polars: 24 points
  First: DataPoint(timestamp=datetime.datetime(2024, 6, 1, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), value=120.0)
  Missing value preserved: True


## 11. Validation

`validate()` checks for common issues: timestamp ordering and frequency consistency.

In [22]:
# Our time series is well-formed
print("Valid series:", ts.validate())

# Create one with problems
bad_ts = TimeSeries(
    hourly_cet,
    timestamps=[
        base + timedelta(hours=2),  # out of order
        base + timedelta(hours=1),
        base + timedelta(hours=5),  # gap (inconsistent frequency)
    ],
    values=[1.0, 2.0, 3.0],
)

print("\nBad series warnings:")
for w in bad_ts.validate():
    print(f"  - {w}")

Valid series: []

Bad series warnings:
  - timestamps not strictly increasing at index 1: 2024-06-01 02:00:00+00:00 >= 2024-06-01 01:00:00+00:00
  - inconsistent frequency at index 1: expected 1:00:00, got -1 day, 23:00:00


## 12. GeoArea Example

For area-based data (e.g., price zones, weather regions), use `GeoArea` as the
location in metadata.

In [25]:
area_meta = Metadata(
    unit="EUR/MWh",
    data_type=DataType.ACTUAL,
    location=no1,
    name="price",
    description="Day-ahead electricity price for NO1",
)

daily_prices = TimeSeries(
    Resolution(frequency=Frequency.P1D),
    area_meta,
    timestamps=[base + timedelta(days=i) for i in range(7)],
    values=[45.2, 47.8, 42.1, 50.3, 48.9, 44.6, 46.7],
)

print(f"Price area: {area_meta.location.name}")
print(f"Centroid:   {no1.centroid}")
print()
daily_prices.to_pandas_dataframe()

Price area: NO1
Centroid:   GeoLocation(latitude=60.0, longitude=9.0)



,price
timestamp,
2024-06-01 00:00:00+00:00,45.2
2024-06-02 00:00:00+00:00,47.8
2024-06-03 00:00:00+00:00,42.1
2024-06-04 00:00:00+00:00,50.3
2024-06-05 00:00:00+00:00,48.9
2024-06-06 00:00:00+00:00,44.6
2024-06-07 00:00:00+00:00,46.7


## 13. Performance (vectorisation)

Quick benchmark of the vectorised paths: `to_numpy()`, scalar arithmetic, `to_pandas_dataframe()`, and pandas round-trip. Uses `timeit` for stable timings.

In [5]:
import timeit
from datetime import datetime, timedelta, timezone
from timedatamodel import TimeSeries, Resolution, Metadata, Frequency, DataType

def make_series(n: int, pct_none: float = 0.05):
    """TimeSeries of n points with ~pct_none missing."""
    base = datetime(2024, 1, 1, tzinfo=timezone.utc)
    res = Resolution(frequency=Frequency.PT1H)
    meta = Metadata(name="value", data_type=DataType.ACTUAL)
    timestamps = [base + timedelta(hours=i) for i in range(n)]
    values = [120.0 + i * 0.1 if (i % int(1 / pct_none)) else None for i in range(n)]
    return TimeSeries(res, meta, timestamps=timestamps, values=values)

sizes = [10_000, 100_000, 500_000]
print("Vectorised TimeSeries timings (best of 5 runs)\n")
print(f"{'N':>10} | {'to_numpy()':>12} | {'ts+1':>10} | {'to_pandas':>12} | {'from_pandas':>12}")
print("-" * 62)

for n in sizes:
    ts = make_series(n)
    df = ts.to_pandas_dataframe()
    t_np = timeit.repeat(lambda: ts.to_numpy(), number=10, repeat=5)
    t_add = timeit.repeat(lambda: ts + 1.0, number=10, repeat=5)
    t_pd = timeit.repeat(lambda: ts.to_pandas_dataframe(), number=10, repeat=5)
    t_fp = timeit.repeat(lambda: TimeSeries.from_pandas(df, ts.resolution, ts.metadata), number=10, repeat=5)
    print(f"{n:>10,} | {min(t_np)*100:>10.2f}ms | {min(t_add)*100:>8.2f}ms | {min(t_pd)*100:>10.2f}ms | {min(t_fp)*100:>10.2f}ms")

print("\nDone. Scale N up (e.g. 1_000_000) to stress-test.")

Vectorised TimeSeries timings (best of 5 runs)

         N |   to_numpy() |       ts+1 |    to_pandas |  from_pandas
--------------------------------------------------------------
    10,000 |       0.58ms |     0.64ms |       1.45ms |       1.51ms
   100,000 |       5.27ms |     6.84ms |      14.35ms |      15.87ms
   500,000 |      27.78ms |    38.14ms |      74.75ms |      83.69ms

Done. Scale N up (e.g. 1_000_000) to stress-test.
